# PART 3 — Model Evaluation + Metrics Analysis + Prediction Visualization

## 3D Kidney and Kidney Tumor Segmentation Using Deep Learning

This notebook performs:

- Load trained 3D U-Net model from PART 2
- Test dataset inference
- Generate segmentation predictions
- Calculate Dice, IoU, Precision, Recall
- Calculate HD95 boundary metric
- Generate evaluation tables
- Create dissertation figures:
  - Kidney vs Tumour Dice Comparison
  - Metrics Comparison
  - HD95 Comparison
  - CT Image + Ground Truth + 3D U-Net Prediction



In [ ]:

# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nibabel as nib

import torch
import torch.nn as nn

from tqdm.auto import tqdm

from scipy.spatial.distance import directed_hausdorff

print("Libraries loaded")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)


In [ ]:

# ==========================================
# 2. PROJECT PATHS
# ==========================================

PROJECT_PATH = r"C:\Users\USER\Kidney_Segmentation_Project"

DATA_PATH = os.path.join(
    PROJECT_PATH,
    "data"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

RESULT_PATH = os.path.join(
    PROJECT_PATH,
    "results"
)

FIGURE_PATH = os.path.join(
    PROJECT_PATH,
    "figures"
)

BEST_MODEL = os.path.join(
    MODEL_PATH,
    "best_3D_UNet_model.pt"
)

os.makedirs(RESULT_PATH, exist_ok=True)
os.makedirs(FIGURE_PATH, exist_ok=True)

print(BEST_MODEL)


In [ ]:

# ==========================================
# 3. LOAD MODEL ARCHITECTURE
# ==========================================

# Import the same UNet3D architecture from PART 2

# NOTE:
# Copy the UNet3D class from PART 2 here
# before loading weights.

print("Placed UNet3D architecture here from PART 2")


In [ ]:

# ==========================================
# 4. LOAD TRAINED MODEL
# ==========================================

checkpoint = torch.load(
    BEST_MODEL,
    map_location=DEVICE
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(DEVICE)

model.eval()

print(
    "Model loaded successfully"
)

print(
    "Best validation Dice:",
    checkpoint.get("best_val_dice","Not saved")
)


In [ ]:

# ==========================================
# 5. METRIC FUNCTIONS
# ==========================================

def dice_score_binary(pred, true):

    intersection = np.sum(
        pred * true
    )

    return (
        2.0 * intersection
        /
        (
            np.sum(pred)
            +
            np.sum(true)
            +
            1e-8
        )
    )


def iou_score(pred,true):

    intersection = np.logical_and(
        pred,
        true
    ).sum()

    union = np.logical_or(
        pred,
        true
    ).sum()

    return intersection/(union+1e-8)



def hd95(pred,true):

    pred_points = np.argwhere(pred)

    true_points = np.argwhere(true)

    if len(pred_points)==0 or len(true_points)==0:
        return np.nan

    forward = directed_hausdorff(
        pred_points,
        true_points
    )[0]

    backward = directed_hausdorff(
        true_points,
        pred_points
    )[0]

    return np.percentile(
        [forward,backward],
        95
    )


In [ ]:

# ==========================================
# 6. TEST CASE LOADING
# ==========================================

test_cases = sorted(
    [
        os.path.join(DATA_PATH,x)
        for x in os.listdir(DATA_PATH)
        if x.startswith("case_")
    ]
)

test_cases = test_cases[-110:]

print(
    "Testing Cases:",
    len(test_cases)
)


In [ ]:

# ==========================================
# 7. SAVE EVALUATION RESULTS
# ==========================================

results=[]


# This loop expects the same patch inference
# function created in PART 2.

for case in tqdm(test_cases):

    print(
        "Evaluate:",
        case
    )

    # Load CT
    # Generate prediction
    # Compare with ground truth

    # Replace these placeholders
    # with reconstructed prediction output.

    result = {

        "Case":os.path.basename(case),

        "Dice":0,

        "Kidney Dice":0,

        "Tumour Dice":0,

        "IoU":0,

        "Precision":0,

        "Recall":0,

        "HD95":0

    }

    results.append(result)


df_results = pd.DataFrame(results)

df_results.to_csv(
    os.path.join(
        RESULT_PATH,
        "evaluation_metrics.csv"
    ),
    index=False
)

df_results.head()


In [ ]:

# ==========================================
# 8. MODEL PERFORMANCE GRAPH
# ==========================================

summary = {

"Dice":df_results["Dice"].mean(),

"IoU":df_results["IoU"].mean(),

"Precision":df_results["Precision"].mean(),

"Recall":df_results["Recall"].mean()

}


plt.figure(figsize=(8,5))

plt.bar(
    summary.keys(),
    summary.values()
)

plt.ylabel(
    "Score"
)

plt.title(
    "3D U-Net Performance Metrics"
)

plt.ylim(0,1)

plt.grid(axis="y")

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "Performance_Metrics.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# ==========================================
# 9. KIDNEY VS TUMOUR DICE GRAPH
# ==========================================

values = [

df_results["Kidney Dice"].mean(),

df_results["Tumour Dice"].mean()

]


plt.figure(figsize=(7,5))

plt.bar(
    ["Kidney","Tumour"],
    values
)

plt.ylabel(
    "Dice Score"
)

plt.title(
    "Kidney vs Tumour Segmentation Dice"
)

plt.ylim(0,1)

plt.grid(axis="y")

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "Kidney_vs_Tumour_Dice.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# ==========================================
# 10. HD95 GRAPH
# ==========================================

plt.figure(figsize=(7,5))

plt.bar(
    ["3D U-Net"],
    [df_results["HD95"].mean()]
)

plt.ylabel(
    "HD95 (mm)"
)

plt.title(
    "Boundary Error Evaluation"
)

plt.grid(axis="y")

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "HD95_Error.png"
    ),
    dpi=600,
    bbox_inches="tight"
)

plt.show()


In [ ]:

# ==========================================
# 11. CT + GT + PREDICTION FIGURE
# ==========================================

# Use one selected test case

print(
    "Add reconstructed CT volume,"
    " ground truth mask and prediction here"
)

# Output format:

# (a) CT Image
# (b) Ground Truth
# (c) 3D U-Net Prediction
